# 2402-01 · One bad query, three symptoms · solution

The reference version of the notebook you build with the lab guide, cell by cell. Open it if you get stuck.

**The question:** 2024 revenue per customer, tier and product category.

| Run | What changes | Files pruned |
|---|---|---|
| **A** | the bad query: function on the date column | 0 |
| **B** | instruction 1 from Genie Code: filter on the raw column | still 0: the layout does not help yet |
| **C** | instruction 2: `orders` clustered by date, same query as B | most of 2023 |

The heavy shuffle and the skew are **read** in the Query Profile of A: they come from the joins and the data, not from the filter.

In [ ]:
%run "./2402-00-create-environment"

## 1 · Measure first: run A and open its Query Profile

In [ ]:
QUERY_A = """
-- A · the bad query, as someone wrote it
SELECT c.customer_id, c.tier, p.category,
       SUM(oi.quantity * p.list_price) AS revenue,
       COUNT(*)                        AS line_items
FROM lab2402.shop.orders o
JOIN lab2402.shop.order_items oi ON o.order_idx = oi.order_idx
JOIN lab2402.shop.products p     ON oi.product_idx = p.product_idx
JOIN lab2402.shop.customers c    ON o.customer_idx = c.customer_idx
WHERE date_format(o.order_date, 'yyyy-MM') BETWEEN '2024-01' AND '2024-12'
GROUP BY c.customer_id, c.tier, p.category
ORDER BY revenue DESC
"""

display(spark.sql(QUERY_A))

## 2 · Benchmark A

In [ ]:
benchmark("A", QUERY_A)

## 3 · Instruction 1: fix the predicate

In [ ]:
QUERY_B = """
-- B · instruction 1: the filter on the raw column, nothing else changes
SELECT c.customer_id, c.tier, p.category,
       SUM(oi.quantity * p.list_price) AS revenue,
       COUNT(*)                        AS line_items
FROM lab2402.shop.orders o
JOIN lab2402.shop.order_items oi ON o.order_idx = oi.order_idx
JOIN lab2402.shop.products p     ON oi.product_idx = p.product_idx
JOIN lab2402.shop.customers c    ON o.customer_idx = c.customer_idx
WHERE o.order_date >= DATE'2024-01-01' AND o.order_date < DATE'2025-01-01'
GROUP BY c.customer_id, c.tier, p.category
ORDER BY revenue DESC
"""

display(spark.sql(QUERY_B))

In [ ]:
benchmark("B", QUERY_B)

## 4 · Instruction 2: fix the data layout

In [ ]:
spark.sql(f"ALTER TABLE lab2402.shop.orders SET TBLPROPERTIES ('delta.targetFileSize' = '{LAYOUT_FILE_SIZE}')")
spark.sql("ALTER TABLE lab2402.shop.orders CLUSTER BY (order_date)")
spark.sql("OPTIMIZE lab2402.shop.orders FULL")      # recluster the existing data, not only new files
file_stats("orders")

In [ ]:
display(spark.sql(QUERY_B))   # the same query as B, on the new layout

In [ ]:
benchmark("C", QUERY_B)

## 5 · The proof: A / B / C from `system.query.history`

In [ ]:
last_runs()

**How to read it**

| Column | What it tells you |
|---|---|
| `read_files`, `pruned_files`, `read_mb` | A and B prune nothing; C skips most of 2023 |
| `shuffle_mb` | The heavy shuffle: it shrinks in C only because fewer rows reach the joins |
| `task_time_ms` | The **cost**: the sum of all task time |
| `wall_clock_ms` | What the user **waits** |
| `any_cache_hit` | Must be 0: otherwise the result cache answered and the benchmark is not honest |